# Lab 11: Grid Localization using Bayes Filter (Real Robot)

### <span style="color:rgb(0,150,0)">It is recommended that you close any heavy-duty applications running on your system while working on this lab.</span>

### <span style="color:rgb(0,150,0)">The notebook only provides skeleton code for you to integrate the Localization class with the Real Robot.</span>

<hr>

In [1]:
%load_ext autoreload
%autoreload 2

import traceback
from notebook_utils import *
from Traj import *
import asyncio
import pathlib
import os
from utils import load_config_params
from localization_extras import Localization
import threading

# The imports below will only work if you copied the required ble-related python files 
# into the notebooks directory
from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import numpy as np

# Setup Logger
LOG = get_logger('demo_notebook.log')
LOG.propagate = False

# Init GUI and Commander
gui = GET_GUI()
cmdr = gui.launcher.commander

/home/leslier/programming/ece5160-fast-robotics/FastRobots_ble/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


2026-04-22 15:15:47,055 | INFO     |: Logger demo_notebook.log initialized.


In [2]:
# Start the plotter
START_PLOTTER()

# The RealRobot class
Define the RealRobot class in the code cell below, based on the documentation and your real robot communication protocol. <br>
This class is used by the **Localization** class to communicate with the real robot. <br>
More specifically, the **Localization** class utilizes the **RealRobot's** member function **perform_observation_loop()** to get the 18 sensor readings and store them in its member variable **obs_range_data**, which is then utilized in the update step.

In [27]:
def connectBLE(ble, timeout_s=10.0):
    # Fast path: already connected
    try:
        if ble.is_connected():
            return True
    except Exception:
        pass

    # Reload config ONCE (important)
    ble.reload_config()

    t0 = time.time()
    while time.time() - t0 < timeout_s:
        try:
            ble.connect()
        except Exception as e:
            # If it says already connected, treat as success
            if "already connected" in str(e).lower():
                return True

        # Give the stack a moment, then re-check
        time.sleep(0.2)
        try:
            if ble.is_connected():
                return True
        except Exception:
            pass

        time.sleep(0.3)

    raise TimeoutError("Failed to connect to BLE device")

def tiles_to_world(tx_tiles, ty_tiles):
    TILE_SIZE_M     = 0.3048   # 1 foot per tile
    x_r = tx_tiles * TILE_SIZE_M
    y_r = ty_tiles * TILE_SIZE_M
    return x_r, y_r

class RealRobot():
    def __init__(self, commander, ble):
        # Load world config
        self.world_config = os.path.join(str(pathlib.Path(os.getcwd()).parent), "config", "world.yaml")
        
        self.config_params = load_config_params(self.world_config)
        
        # Commander to commuincate with the Plotter process
        # Used by the Localization module to plot odom and belief
        self.cmdr = commander

        # ArtemisBLEController to communicate with the Robot
        self.ble = ble

        self.yaw_values = []
        self.dist_values = []
        self.status_log = []
        self.done = threading.Event()

    def get_pose(self):
        """Get robot pose based on odometry
        
        Returns:
            current_odom -- Odometry Pose (meters, meters, degrees)
        """
        raise NotImplementedError("get_pose is not implemented")

    def _mapping_notification_callback(self, uuid, data: bytearray):
        msg = self.ble.bytearray_to_string(data).strip()
        parts = [p.strip() for p in msg.split(",") if p.strip()]
    
        for token in parts:
            if token.lower() == "end":
                print("End of mapping data")
                self.done.set()
                return
    
            if token.startswith("STATUS:"):
                self.status_log.append(token)
                print(f"  {token}")
                continue
    
            try:
                yaw_str, dist_str = token.split(":", 1)
                self.yaw_values.append(float(yaw_str))
                self.dist_values.append(float(dist_str))
            except ValueError:
                print("Bad token:", repr(token))
    
    def _get_mapping_data(self, timeout_s=120.0):
        self.yaw_values.clear()
        self.dist_values.clear()
        self.status_log.clear()
        self.done.clear()
    
        self.ble.send_command(CMD.COLLECT_MAPPING_DATA, "")
    
        t0 = time.time()
        while not self.done.is_set() and (time.time() - t0) < timeout_s:
            self.ble.sleep(0.01)
    
        if not self.done.is_set():
            print("Timeout reached.")
        return list(self.yaw_values), list(self.dist_values)
    
    def perform_observation_loop(self, rot_vel=120):
        """Perform the observation loop behavior on the real robot, where the robot does  
        a 360 degree turn in place while collecting equidistant (in the angular space) sensor
        readings, with the first sensor reading taken at the robot's current heading. 
        The number of sensor readings depends on "observations_count"(=18) defined in world.yaml.
        
        Keyword arguments:
            rot_vel -- (Optional) Angular Velocity for loop (degrees/second)
                        Do not remove this parameter from the function definition, even if you don't use it.
        Returns:
            sensor_ranges   -- A column numpy array of the range values (meters)
            sensor_bearings -- A column numpy array of the bearings at which the sensor readings were taken (degrees)
                               The bearing values are not used in the Localization module, so you may return a empty numpy array
        """

        # --- Run ---
        try:
            connectBLE(self.ble)
            self.ble.start_notify(self.ble.uuid['RX_STRING'], self._mapping_notification_callback)
        except Exception as e:
            if "Notify acquired" in str(e):
                print("Notify already active; continuing.")
            else:
                raise
        time.sleep(0.2)
        
        yaws, dists = self._get_mapping_data()
        
        try:
            self.ble.stop_notify(self.ble.uuid['RX_STRING'])
        except Exception as e:
            print("Failed to stop notifications with exception: ", e)
        
        print(f"\nCollected {len(yaws)} points")

        #sensor_ranges = np.array(dists).reshape(-1, 1) / 1000.0  # mm → meters
        #sensor_bearings = np.array([-y for y in yaws]).reshape(-1, 1) # make yaws negative since my robot spins cw but world is ccw
        sensor_ranges = np.array(dists).reshape(-1, 1) / 1000.0
        sensor_bearings = np.array(yaws).reshape(-1, 1)
        return sensor_ranges, sensor_bearings

In [4]:
# Get ArtemisBLEController object
ble = get_ble_controller()

# Connect to the Artemis Device
#ble.connect()

In [7]:
# Initialize RealRobot with a Commander object to communicate with the plotter process
# and the ArtemisBLEController object to communicate with the real robot
robot = RealRobot(cmdr, ble)

# Initialize mapper
# Requires a VirtualRobot object as a parameter
mapper = Mapper(robot)

# Initialize your BaseLocalization object
# Requires a RealRobot object and a Mapper object as parameters
loc = Localization(robot, mapper)

## Plot Map
cmdr.plot_map()

2026-04-22 15:27:34,447 | INFO     |:  | Number of observations per grid cell: 30
2026-04-22 15:27:34,448 | INFO     |:  | Precaching Views...
2026-04-22 15:27:37,135 | INFO     |:  | Precaching Time: 2.686 secs
2026-04-22 15:27:37,136 | INFO     |: Initializing beliefs with a Uniform Distribution
2026-04-22 15:27:37,137 | INFO     |: Uniform Belief with each cell value: 0.00051440329218107


# Run an update step of the Bayes Filter

In [41]:
# Reset Plots
cmdr.reset_plotter()

x_tile = 0
y_tile = 0


# Init Uniform Belief
loc.init_grid_beliefs()

# Get Observation Data by executing a 360 degree rotation motion
loc.get_observation_data()

# Run Update Step
loc.update_step()
loc.plot_update_step_data(plot_data=True)

# Plot Odom and GT
# current_odom, current_gt = robot.get_pose()
x_world, y_world = tiles_to_world(x_tile, y_tile)
cmdr.plot_gt(x_world, y_world)
# cmdr.plot_odom(current_odom[0], current_odom[1])

2026-04-22 16:52:47,313 | INFO     |: Initializing beliefs with a Uniform Distribution
2026-04-22 16:52:47,314 | INFO     |: Uniform Belief with each cell value: 0.00051440329218107
2026-04-22 16:52:47,317 | INFO     |: Looking for Artemis Nano Peripheral Device: c0:42:d1:79:ab:49
2026-04-22 16:52:47,318 | INFO     |: Scanning for device with address: c0:42:d1:79:ab:49, service UUID: 4843fd62-068c-4a37-af34-e724c6a05681
2026-04-22 16:52:57,512 | INFO     |: Found 207 total devices
2026-04-22 16:52:57,515 | INFO     |: Found matching device: C0:42:D1:79:AB:49 (name: Artemis BLE)
2026-04-22 16:52:58,471 | INFO     |: Connected to c0:42:d1:79:ab:49
  STATUS:COLLECT:y=358.254120:sp=0.000000:e=1.745880:pts=0
  STATUS:TURN:y=355.446106:sp=-10.000000:e=-5.446106:pts=2
  STATUS:TURN:y=335.972687:sp=-20.000000:e=4.027313:pts=3
  STATUS:TURN:y=334.491791:sp=-20.000000:e=5.508209:pts=3
  STATUS:TURN:y=334.297089:sp=-30.000000:e=-4.297089:pts=4
  STATUS:COLLECT:y=316.937256:sp=-40.000000:e=3.06274